In [64]:
pip install mss

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import time
import pyautogui

try:
    import mss
except ImportError:
    print("❌ 找不到 mss 套件！請先在終端機輸入: pip install mss")
    exit()

try:
    import win32api, win32con
    USE_WIN32 = True
except ImportError:
    USE_WIN32 = False

# ================= 你的寶石中心座標 =================
TOP_LEFT_X = 408      
TOP_LEFT_Y = 490      
BOTTOM_RIGHT_X = 759  
BOTTOM_RIGHT_Y = 839  
# ===================================================

pyautogui.FAILSAFE = False
pyautogui.PAUSE = 0.0

STEP_X = (BOTTOM_RIGHT_X - TOP_LEFT_X) / 7.0
STEP_Y = (BOTTOM_RIGHT_Y - TOP_LEFT_Y) / 7.0

SCREEN_XY = []
PAD = int(STEP_X * 0.4)
REG_X, REG_Y = int(TOP_LEFT_X - PAD), int(TOP_LEFT_Y - PAD)
REG_W = int((BOTTOM_RIGHT_X - TOP_LEFT_X) + PAD * 2)
REG_H = int((BOTTOM_RIGHT_Y - TOP_LEFT_Y) + PAD * 2)

R_OUT = int(STEP_X * 0.24)
RING_OFFSETS = [
    (-R_OUT, 0), (R_OUT, 0), (0, -R_OUT), (0, R_OUT),
    (-int(R_OUT*0.7), -int(R_OUT*0.7)), (int(R_OUT*0.7), -int(R_OUT*0.7)),
    (-int(R_OUT*0.7), int(R_OUT*0.7)), (int(R_OUT*0.7), int(R_OUT*0.7))
]

REL_XY = []
for r in range(8):
    for c in range(8):
        sx = int(TOP_LEFT_X + c * STEP_X)
        sy = int(TOP_LEFT_Y + r * STEP_Y)
        SCREEN_XY.append((sx, sy))
        REL_XY.append((sx - REG_X, sy - REG_Y))

SCT = mss.mss()
MONITOR = {"top": REG_Y, "left": REG_X, "width": REG_W, "height": REG_H}

COLUMN_COOLDOWN = [0.0] * 8
DROP_TIME = 0.17  # 穩定抗噪掉落時間

def scan_board():
    img = SCT.grab(MONITOR)
    board = [0] * 64
    special = [False] * 64
    current_time = time.perf_counter()

    for idx in range(64):
        x_col = idx % 8
        if current_time < COLUMN_COOLDOWN[x_col]:
            continue

        cx, cy = REL_XY[idx]
        center_px = img.pixel(cx, cy)
        
        ring = [img.pixel(cx + dx, cy + dy) for dx, dy in RING_OFFSETS]
        ring.sort(key=lambda p: p[0] + p[1] + p[2])
        mid = ring[2:6]
        
        r = (mid[0][0] + mid[1][0] + mid[2][0] + mid[3][0]) >> 2
        g = (mid[0][1] + mid[1][1] + mid[2][1] + mid[3][1]) >> 2
        b = (mid[0][2] + mid[1][2] + mid[2][2] + mid[3][2]) >> 2

        mx = max(r, g, b)
        mn = min(r, g, b)
        diff = mx - mn

        if mx < 75: continue 
        if diff < 38 and mx > 115:
            board[idx] = 6
            continue

        if (center_px[0] > 210 and center_px[1] > 210 and center_px[2] > 210) or (ring[-1][0]+ring[-1][1]+ring[-1][2] > 680):
            special[idx] = True

        if diff == 0: continue

        if mx == r: h = (60 * (g - b) // diff) % 360
        elif mx == g: h = 60 * (b - r) // diff + 120
        else: h = 60 * (r - g) // diff + 240

        if h < 13 or h >= 338: board[idx] = 1
        elif h < 39: board[idx] = 2
        elif h < 70: board[idx] = 3
        elif h < 165: board[idx] = 4
        elif h < 250: board[idx] = 5
        else: board[idx] = 7

    return board, special

def calculate_match_score(b, idx):
    if b[idx] == 0: return 0
    val = b[idx]
    x, y = idx % 8, idx // 8
    
    h_count, v_count = 1, 1
    for i in range(x - 1, -1, -1):
        if b[y * 8 + i] == val: h_count += 1
        else: break
    for i in range(x + 1, 8):
        if b[y * 8 + i] == val: h_count += 1
        else: break
        
    for i in range(y - 1, -1, -1):
        if b[i * 8 + x] == val: v_count += 1
        else: break
    for i in range(y + 1, 8):
        if b[i * 8 + x] == val: v_count += 1
        else: break
        
    score = 0
    if h_count >= 5 or v_count >= 5: score += 50000 
    elif h_count >= 3 and v_count >= 3: score += 20000
    elif h_count >= 4 or v_count >= 4: score += 8000
    else:
        if h_count >= 3: score += 10
        if v_count >= 3: score += 10
    return score

def find_best_moves(board, special):
    valid_moves = []
    for y in range(8):
        for x in range(8):
            i = y * 8 + x
            if board[i] == 0: continue
            
            if x < 7:
                right = i + 1
                if board[right] != 0 and board[i] != board[right]:
                    board[i], board[right] = board[right], board[i]
                    score = calculate_match_score(board, i) + calculate_match_score(board, right)
                    board[i], board[right] = board[right], board[i]
                    if score > 0:
                        depth_bonus = (y ** 2) * 15
                        special_bonus = 15000 if (special[i] or special[right]) else 0
                        valid_moves.append((score + depth_bonus + special_bonus, i, right))
                        
            if y < 7:
                down = i + 8
                if board[down] != 0 and board[i] != board[down]:
                    board[i], board[down] = board[down], board[i]
                    score = calculate_match_score(board, i) + calculate_match_score(board, down)
                    board[i], board[down] = board[down], board[i]
                    if score > 0:
                        depth_bonus = (y ** 2) * 15
                        special_bonus = 15000 if (special[i] or special[down]) else 0
                        valid_moves.append((score + depth_bonus + special_bonus, i, down))
                        
    valid_moves.sort(key=lambda m: m[0], reverse=True)
    return valid_moves

def fast_drag(x1, y1, x2, y2):
    """10ms 黃金滑動對齊"""
    if USE_WIN32:
        win32api.SetCursorPos((x1, y1))
        win32api.mouse_event(win32con.MOUSEEVENTF_LEFTDOWN, 0, 0, 0, 0)
        time.sleep(0.004)
        win32api.SetCursorPos((x2, y2))
        time.sleep(0.006)
        win32api.mouse_event(win32con.MOUSEEVENTF_LEFTUP, 0, 0, 0, 0)
    else:
        pyautogui.moveTo(x1, y1)
        pyautogui.mouseDown()
        time.sleep(0.008)
        pyautogui.moveTo(x2, y2, duration=0.008)
        pyautogui.mouseUp()

def play_bot(run_time_seconds=70):
    print(f"🎯 修正完成！Bejeweled Blitz 穩定消除版啟動 ({run_time_seconds} 秒)...")
    
    start_time = time.perf_counter()
    end_time = start_time + run_time_seconds
    attempts = 0

    while time.perf_counter() < end_time:
        board, special = scan_board()
        moves = find_best_moves(board, special)
        
        if moves:
            score, idx1, idx2 = moves[0]
            col1, col2 = idx1 % 8, idx2 % 8
            
            x1, y1 = SCREEN_XY[idx1]
            x2, y2 = SCREEN_XY[idx2]
            
            fast_drag(x1, y1, x2, y2)
            attempts += 1
            
            current_time = time.perf_counter()
            COLUMN_COOLDOWN[col1] = current_time + DROP_TIME
            COLUMN_COOLDOWN[col2] = current_time + DROP_TIME
            
            time.sleep(0.002)
        else:
            time.sleep(0.004)

    actual_run_time = time.perf_counter() - start_time
    print(f"\n🏆 遊戲結束！執行了 {attempts} 次有效操作！")

if __name__ == "__main__":
    print("✅ 誤判點擊已全面修復，準備恢復 20 萬+ 實力！")
    print("請在 3 秒內切換至遊戲畫面...")
    time.sleep(3)
    play_bot(run_time_seconds=70)